# Portfolio Construction (Black–Litterman & Factors) — Theory

*Companion to **Portfolio Construction Black Litterman — Code.ipynb**.*

This notebook collects the **conceptual and mathematical background** for the project,
separated from the implementation. It contains no code and no result-specific findings —
those live in the Code notebook alongside the cells that produce them. Sections here mirror
the structure of the Code notebook so the two can be read side by side.

---

## Contents
1. Factor Analysis
2. Tactical Allocation: Factors vs Macro Data
3. Black–Litterman Integration
4. Portfolio Optimizations
5. Performance & Risk Analytics


---

# 1. Factor Analysis

#### Factor Regression with Market

##### Factor Orthogonality and Redundancy

The primary goal of regressing secondary factors (SMB, HML, MOM) against the Market (MKT_RF) is to determine if these factors provide unique risk premia or if they are simply "Market Beta in disguise.

Key Metric: A low R-squared (typically <0.15) across these regressions suggests that the factors are largely orthogonal (independent) to the market. This is a positive finding for our portfolio construction, as it confirms that adding these factors provides true diversification rather than just doubling down on market direction.

##### Market Sensitivity (The Betas)

HML/SMB Betas: If the Beta for HML or SMB is near zero and statistically insignificant (low T-stat), it indicates these factors are "Market Neutral" by construction during the training period.

Cyclicality: A significantly positive Beta for a factor would suggest it is "Pro-cyclical" (moves with the market), while a negative Beta suggests "Defensive" characteristics. For instance, Momentum (MOM) often exhibits negative market beta during sharp trend reversals.

##### Statistical Significance (T-Stats and P-Values)

We evaluate the robustness of these relationships using the T-stats.

Significant Relationships (∣T∣>2): Any factor with a high T-stat implies a reliable historical link to market movements.

Insignificant Alphas: Ideally, the Intercept (Alpha) of these regressions should be positive. A significant positive Alpha indicates that the factor produced excess returns during the training period that cannot be explained by market exposure alone—justifying its inclusion as an "Alpha-seeking" asset in our tactical model.


#### Factor Rolling Regression with MKT

The rolling regression helps us understand how factors behave with MKT over a rolling window of 1 year (252 trading days).

---

# 2. Tactical Allocation: Factors vs Macro Data

### Macroeconomic Data Acquisition: Foundation for Tactical View Generation

> ⚠️ **Describes the superseded linear-regression approach.**
> Retained because the Code notebook still contains that section for documentation.
> The theory actually underlying the views now used is in
> **Regime-Conditional View Generation** below.

#### Purpose and Rationale

This section establishes the macroeconomic foundation for generating tactical views in our Black-Litterman framework. The strategic decision to pull macro data **one year before the training period** is critical for several reasons:

1. **Lag Requirements for Derived Indicators**: 
   - **CPI Year-over-Year Calculation**: The Consumer Price Index (CPI) is transformed into a year-over-year percentage change, which requires 12 months of historical data to compute the first valid observation. By starting data extraction one year earlier, we ensure that the first CPI YoY value can be calculated at the beginning of our training period (2021-01-01), preventing data loss at the start of our analysis window.

2. **Rolling Window Calculations**:
   - The tactical allocation framework relies on **rolling Z-score normalization** (252-day window) to transform raw macro variables into standardized signals. Starting data collection earlier provides sufficient historical context for meaningful rolling statistics from the first day of the training period.

3. **Data Continuity and Alignment**:
   - Macroeconomic data from different sources (FRED) may have varying frequencies (daily vs. monthly) and potential data gaps. The extended extraction period allows for proper forward-filling of monthly CPI data to daily frequency while maintaining temporal alignment with factor returns data.

#### Macroeconomic Indicators and Their Role

The three selected indicators serve distinct but complementary roles in tactical view generation:

- **T10Y2Y (Yield Curve Slope)**: Measures the difference between 10-year and 2-year Treasury yields, serving as a leading indicator of economic expectations and monetary policy outlook. A steepening curve often signals economic expansion expectations, while flattening/inversion suggests recession concerns.

- **CPIAUCSL (Consumer Price Index)**: Transformed into year-over-year inflation rate, this indicator captures price stability conditions. High inflation environments typically favor value stocks (HML factor) and may disadvantage growth-oriented investments.

- **VIXCLS (Volatility Index)**: Represents market expectations of near-term volatility. Elevated VIX levels often coincide with flight-to-quality behavior, benefiting large-cap stocks (negative SMB exposure) and reducing risk-taking behavior.

#### Data Processing Strategy

The forward-filling and backward-filling approach for CPI data is necessary because:
- CPI is published monthly, but factor returns are daily
- The tactical allocation framework requires daily signals for consistency with portfolio rebalancing frequencies
- Forward-filling preserves the most recent CPI value until the next monthly release, maintaining signal continuity
- Backward-filling handles any initial gaps at the beginning of the dataset

#### Connection to Tactical View Generation

These macroeconomic signals are processed through the `process_macro_signals()` function, which:
1. Transforms raw indicators into normalized Z-scores using rolling statistics

The processed macro economic data is aligned for index with factor data, split into training period using the same train start and end date.
the `generate_tactical_views()` function:

1. Creates a "current state" representation by averaging recent signals (63-day consensus window)
2. Feeds into the regression framework that translates macro conditions into expected factor returns

The resulting Q vector for Black-Litterman reflects how current economic conditions—as captured by these three indicators—should influence portfolio tilts toward Size (SMB), Value (HML), and Momentum (MOM) factors.

#### Strategic Importance

This macro data foundation is essential because it:
- **Enables Quantitative Views**: Rather than relying on subjective analyst forecasts, our tactical views are generated systematically from observable economic conditions
- **Provides Forward-Looking Signals**: Macro indicators often lead equity market performance, making them valuable for tactical allocation
- **Ensures Robustness**: Using well-established, publicly available indicators (FRED) ensures transparency and reproducibility of the tactical allocation process

The careful data preparation and extended extraction period ensure that our tactical view generation mechanism has all necessary inputs from day one of the training period, maintaining consistency throughout the backtest and enabling reliable out-of-sample evaluation.

### Regime-Conditional View Generation

#### The problem with a linear predictive regression

The natural first approach to macro-driven views is to regress factor returns on macro variables and
read expected returns off the fitted equation:

$$r_{f,t} = \alpha_f + \sum_k \beta_{f,k} x_{k,t-1} + \varepsilon_{f,t}$$

Three assumptions are embedded here, and each is questionable for this application.

**Constant slope.** The specification asserts that a one-unit move in $x_k$ shifts expected factor
return by $\beta_{f,k}$ identically in every state of the world. If the premise motivating the
exercise is that factor premia are *regime-dependent*, this assumption contradicts the premise.

**Independent observations.** Inference treats each row as an independent draw. Macro variables are
highly persistent, so this is badly violated. For a regressor with first-order autocorrelation
$\rho$, the effective number of independent observations is approximately

$$N_{eff} \approx N \cdot \frac{1-\rho}{1+\rho}$$

At daily frequency with $\rho \approx 0.99$, a sample of several hundred rows carries only a
handful of independent observations. Reported standard errors are correspondingly optimistic.

**Levels carry information.** Regressing returns on the published *level* of a macro series tests
whether public information predicts returns. Under even weak market efficiency this should fail;
what moves prices is the unanticipated component.

#### The alternative: conditional means within discrete states

Rather than fitting a response surface, partition history into a small number of macro **regimes**
and estimate the mean factor return within each:

$$Q_f = \hat{\mathbb{E}}\big[r_f \mid s_t = s^*\big] = \frac{1}{n_{s^*}}\sum_{t: s_t = s^*} r_{f,t}$$

where $s_t$ is the regime label at time $t$ and $s^*$ the regime prevailing at the view-formation
date. This estimates two to four numbers per factor instead of a slope vector, imposes no functional
form, and conditions on state rather than on level.

#### Prior, view, and the regime tilt

Separating the unconditional and conditional moments makes the information content of a view
explicit:

| Quantity | Meaning |
|---|---|
| $\pi_f$ | Unconditional mean of factor $f$ over the calibration window |
| $Q_f$ | Conditional mean given the prevailing regime |
| $Q_f - \pi_f$ | The **regime tilt** — what the classification contributes |

A view whose tilt is near zero contains no regime information, however large $Q_f$ may be in
absolute terms. Reporting the tilt alongside the level is therefore more informative than reporting
$Q$ alone.

#### Design requirements on the regime classification

**Real-time admissibility.** Every input must be lagged to its actual publication schedule, and
every threshold must be either a fixed constant or a statistic computed on an expanding window.
A threshold set using full-sample information — a full-sample median, or a cut chosen after
inspecting returns — introduces look-ahead into the classification itself, which is harder to detect
than look-ahead in the returns.

**Ex-ante rather than ex-post criteria.** A filter or threshold justified by economics (a central
bank's target, a zero real rate, a trend-growth definition) is a design choice that can be stated in
advance. A threshold estimated from the same returns being modelled uses the data twice. The
distinction generalises well beyond regime work:

> Condition on what was knowable before the returns were observed; never on what was estimated
> from them.

**Pre-committed factor-to-variable assignment.** With $F$ factors and $V$ candidate conditioning
variables there are $F \times V$ conditional means to inspect. Choosing the assignment after seeing
them guarantees an economically plausible story can be constructed for whichever pairing looks best.
Fixing the assignment on mechanism, in writing, before computing any conditional mean is what makes
the resulting statistics interpretable.

#### Regime versus era

A conditional mean pooled across a long sample can confound the regime with the period in which that
regime happened to occur. If a macro state appears predominantly in one era, and factor premia in
that era differed structurally for unrelated reasons, then

$$\hat{\mathbb{E}}[r_f \mid s^*] \ \text{ conflates } \ \mathbb{E}[r_f \mid s^*] \ \text{ with } \ \mathbb{E}[r_f \mid \text{era}]$$

and the two cannot be separated by any refinement of the classifier — there may simply be only one
occurrence of the state.

The diagnostic is to recompute the conditional spread *within* each era. A spread that keeps its
sign in every sub-period supports pooling. A spread that appears only in the pooled sample is an era
effect. Where a state has no within-era variation at all, the conditional estimate is not merely
imprecise but unidentified, and the appropriate response is to restrict the calibration window to a
period over which the structural environment was stable — with the break placed on an exogenous
event rather than on a data-availability boundary.

---

### Conditional Means, Episode Clustering, and View Uncertainty

#### Why months are not observations

A regime persists. A state lasting eighteen consecutive months constitutes one realisation of that
state, not eighteen independent ones: the returns within it are serially dependent and the
conditioning label is constant throughout. The naive standard error of a conditional mean,

$$SE_{naive} = \frac{s(r)}{\sqrt{n_{months}}},$$

therefore overstates precision by roughly the square root of the average episode length.

#### Clustering by episode

Let an **episode** be a maximal contiguous run of months sharing a regime label, and let
$\bar{r}_j$ denote the mean return within episode $j$. Treating episodes as the independent unit:

$$SE_{cluster} = \frac{s(\bar{r}_1, \dots, \bar{r}_{J})}{\sqrt{J}}$$

with $J$ the number of episodes. This is a cluster-robust estimator with clusters defined by regime
occurrence, and it is the appropriate measure of how precisely a conditional mean is known.

The ratio $SE_{cluster} / SE_{naive}$ is a useful diagnostic in its own right: it quantifies exactly
how much apparent confidence the independence assumption manufactures.

#### From standard error to $\Omega$

In Black–Litterman, $\Omega$ encodes uncertainty in the views. Where each view is a conditional mean
and views are treated as independent, the diagonal follows directly:

$$\Omega_{ii} = \big(SE_{cluster,i}\big)^2$$

Three properties recommend this over calibrating $\Omega$ from the covariance matrix.

**Interpretability.** $\Omega_{ii}$ becomes the squared standard error of a quantity that was
actually estimated, rather than a scaled function of asset covariance.

**Automatic caution where evidence is thin.** A regime observed over few episodes yields a large
standard error, hence a wide $\Omega$, hence a posterior that remains near the equilibrium prior.
The framework becomes conservative precisely where the data is weakest, without manual intervention.

**Removal of the $\tau$ dependence.** The common calibration
$\Omega = \text{diag}\big(P\,\tau\,\Sigma\,P'\big)$ makes view uncertainty a function of $\tau$.
If the $\tau$ used to build $\Omega$ differs from the $\tau$ used in the posterior, the algebraic
cancellation that motivates the calibration no longer holds, and the prior/view balance is silently
altered. A measured standard error contains no $\tau$, so the inconsistency cannot arise.

#### Reporting the weight a view receives

For a single view, the posterior expectation is a precision-weighted blend of prior and view. The
share attributable to the view is

$$w_{view} = \frac{\big(P\tau\Sigma P'\big)_{ii}}{\big(P\tau\Sigma P'\big)_{ii} + \Omega_{ii}}$$

This is the most informative single summary of a view's strength. A low value is not a failure of
the framework: it is the framework correctly reporting that the signal does not justify a large
deviation from equilibrium weights. A weak view combined with an honest $\Omega$ is a defensible
result; a weak view combined with a tight $\Omega$ manufactures confidence the data does not contain.

#### On the detectability of factor premia

The difficulty of estimating conditional means should be understood against the difficulty of
estimating unconditional ones. For a strategy with annualised Sharpe ratio $SR$ observed over $T$
years, the $t$-statistic on its mean return is approximately

$$t \approx SR \times \sqrt{T}$$

Equity factor premia carry long-run Sharpe ratios substantially below one, so the span required for
conventional significance runs to decades. Estimates conditioned on a subset of the sample are
noisier still.

Two implications follow. First, an insignificant unconditional premium over a short window is the
expected outcome and not evidence against the premium's existence. Second, modest $t$-statistics on
conditional means are the realistic target; a specification search terminated when a conventional
threshold is crossed produces a reported statistic that no longer means what it appears to.

---


# 3. Black–Litterman Integration

### The Prior Equilibrium


#### Market Capitalization Weight Calculation

##### Market Definition: 13-Stock Portfolio as "The Market"

The portfolio of 13 stocks, weighted by market capitalization, serves as our proxy for "the market" in the Black-Litterman framework. This approach is justified by several considerations:

1. **Representative Sample**: The 13 stocks were selected ex-ante for sector diversity and heterogeneous macro sensitivity across the large-cap universe. (Two early screening filters — pairwise correlation and factor-regression R² — were removed after diagnostic review showed they selected on market beta rather than factor exposure; both regressions are retained in the code notebook as reported diagnostics only.)

2. **Market Cap Weighting as Equilibrium**: Market capitalization weights represent the aggregate investment decisions of all market participants. Under the Capital Asset Pricing Model (CAPM) assumptions, these weights represent the market portfolio—the portfolio that all investors collectively hold in equilibrium.

3. **Reverse Optimization Foundation**: The Black-Litterman framework uses reverse optimization to derive equilibrium returns. Starting from market cap weights assumes that:
   - Current market prices reflect all available information
   - Market participants are in equilibrium
   - The observed portfolio weights are optimal given current return expectations

**Mathematical Foundation:**
The market cap weight for stock $i$ is calculated as:
$$w_i^{mkt} = \frac{MarketCap_i}{\sum_{j=1}^{n} MarketCap_j}$$

Where $MarketCap_i = SharesOutstanding_i \times Price_i$ (using unadjusted prices to ensure accuracy).

##### Factor Weights Set to Zero: Latent Return Drivers

**Why Factors Have Zero Weights:**

Factors (SMB, HML, MOM) are assigned zero market cap weights in the prior equilibrium calculation because:

1. **Factors Are Not Directly Investable**: Factors represent long/short portfolios or risk exposures, not assets that consume capital. They cannot be purchased directly like stocks.

2. **Latent Return Driver Framework**: In this implementation, factors act as **latent return drivers** that influence stock selection and weighting through their correlation structure, but do not require capital allocation.

3. **Institutional Reality**: In practice, factors are accessed through:
   - Smart beta ETFs
   - Long/short factor portfolios
   - Derivatives or swaps
   - But not as direct holdings in a traditional portfolio

**Mathematical Representation:**
The prior weight vector is structured as:
$$\mathbf{w}_{prior} = \begin{bmatrix} \mathbf{w}_{stocks} \\ \mathbf{0}_{factors} \end{bmatrix}$$

Where:
- $\mathbf{w}_{stocks}$: Market cap weights for stocks (sum to 1)
- $\mathbf{0}_{factors}$: Zero weights for factors (sum to 0)

This ensures that factors influence the return vector and covariance structure but do not consume portfolio capital.


---

#### Covariance Matrix Estimation: Conventional vs. Ledoit-Wolf Shrinkage

##### Conventional Sample Covariance

The conventional covariance matrix is calculated using the standard sample covariance estimator:

$$\mathbf{\Sigma}_{conventional} = \frac{1}{T-1} \sum_{t=1}^{T} (\mathbf{r}_t - \bar{\mathbf{r}})(\mathbf{r}_t - \bar{\mathbf{r}})'$$

Where:
- $T$: Number of observations
- $\mathbf{r}_t$: Vector of returns at time $t$
- $\bar{\mathbf{r}}$: Mean return vector

**Properties:**
- Unbiased estimator
- Maximum likelihood estimator under normality
- Problem: High estimation error when number of assets ($n$) is large relative to observations ($T$)
- Problem: Extreme eigenvalues (both very large and very small) lead to unstable portfolio optimization

##### Ledoit-Wolf Shrinkage Covariance

The Ledoit-Wolf estimator addresses estimation error by shrinking the sample covariance toward a structured target:

$$\mathbf{\Sigma}_{LW} = \alpha \mathbf{\Sigma}_{target} + (1-\alpha) \mathbf{\Sigma}_{conventional}$$

Where:
- $\alpha \in [0,1]$: Shrinkage intensity (determined optimally)
- $\mathbf{\Sigma}_{target}$: Structured target matrix (typically single-index model or constant correlation model)

**Optimal Shrinkage Intensity**:
The shrinkage factor $\alpha$ is chosen to minimize the expected quadratic loss:
$$\alpha^* = \arg\min_{\alpha} E[||\mathbf{\Sigma}_{LW} - \mathbf{\Sigma}_{true}||^2]$$

**Benefits**:
1. Reduced Estimation Error: Shrinking toward a structured estimator reduces the impact of sampling noise
2. Improved Conditioning: Prevents extreme eigenvalues, making the matrix more numerically stable
3. Better Out-of-Sample Performance: Shrinkage typically improves portfolio performance in out-of-sample tests
4. Robustness: Less sensitive to outliers and small sample sizes

**Why Use Both:**

1. Comparison and Validation: Comparing results from both methods provides robustness checks and helps identify whether estimation error significantly impacts portfolio construction.

2. Sensitivity Analysis: Understanding how portfolio weights change with different covariance estimates reveals the stability of optimization results.

3. Academic Rigor: Ledoit-Wolf is considered the industry standard for covariance estimation, while conventional covariance provides a baseline for comparison.

4. Practical Considerations: 
   - Conventional: Simpler, interpretable, but may have estimation issues
   - Ledoit-Wolf: More robust, better for optimization, but requires additional computation

**Shrinkage Factor Interpretation:**
- $\alpha \approx 0$: Little shrinkage needed (sample covariance is reliable)
- $\alpha \approx 1$: Heavy shrinkage needed (high estimation error, small sample size)
- Typical values: 0.1 to 0.3 for daily returns with 3+ years of data



#### Risk Aversion Parameter (Lambda) Calculation

##### Market Lambda: Implied Risk Aversion from Market Portfolio

The market risk aversion parameter ($\lambda_{market}$) is derived from the market portfolio using reverse optimization. Under equilibrium, the market portfolio is the tangency portfolio, which implies:

$$\lambda_{market} = \frac{\mu_{market} - r_f}{\sigma_{market}^2}$$

Where:
- $\mu_{market}$: Annualized mean market return
- $\sigma_{market}^2$: Annualized market return variance
- $r_f$: Risk-free rate (assumed 0 for excess returns)

**Derivation:**
Starting from the mean-variance optimization problem:
$$\max_{\mathbf{w}} \mathbf{w}'\boldsymbol{\mu} - \frac{\lambda}{2}\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}$$

The first-order condition for the market portfolio (which is optimal) is:
$$\boldsymbol{\mu} - \lambda \boldsymbol{\Sigma}\mathbf{w}_{mkt} = 0$$

Solving for $\lambda$:
$$\lambda = \frac{\boldsymbol{\mu}'\mathbf{w}_{mkt}}{\mathbf{w}_{mkt}'\boldsymbol{\Sigma}\mathbf{w}_{mkt}} = \frac{\mu_{mkt}}{\sigma_{mkt}^2}$$


**In This Implementation:**
The 13-stock portfolio return series is calculated as the weighted sum of individual stock returns using market capitalization weights:

$$r_{13-stock-portfolio,t} = \sum_{i=1}^{10} w_i^{mkt} \times r_{i,t}$$

Where $w_i^{mkt}$ are the market cap weights for the 13 stocks (summing to 1).

Then lambda is calculated as:

$$\lambda_{market} = \frac{\bar{r}_{13-stock-portfolio} \times 252}{\text{Var}(r_{13-stock-portfolio}) \times 252} = \frac{\bar{r}_{13-stock-portfolio}}{\text{Var}(r_{13-stock-portfolio})}$$

Where $\bar{r}_{13-stock-portfolio}$ is the mean daily return of the 13-stock market portfolio over the training period.

**Why Use the 13-Stock Portfolio Instead of MKT_RF:**

1. **Consistency with Black-Litterman Framework**: The same "market" definition (13-stock portfolio) is used for both reverse optimization and lambda derivation, ensuring internal consistency.

2. **Mathematical Consistency**: The lambda that makes the 13-stock portfolio optimal should be derived from the 13-stock portfolio's own risk-return characteristics, not from a different market proxy.

3. **Portfolio-Specific Risk Aversion**: The 13-stock portfolio may have different risk characteristics than the broad market, and lambda should reflect the specific portfolio's risk-return trade-off.


##### Three Investor Types: Different Risk Aversion Levels

**1. Market Investor ($\lambda_{market}$)**
- **Risk Aversion**: Derived from the 13-stock market portfolio
- **Interpretation**: Represents the "average" investor's risk tolerance for this specific portfolio
- **Portfolio Characteristics**: Baseline equilibrium portfolio
- **Formula**: $\lambda_{market} = \frac{\mu_{13-stock-portfolio}}{\sigma_{13-stock-portfolio}^2}$

**2. Trustee Investor ($\lambda_{trustee} = 2 \times \lambda_{market}$)**
- **Risk Aversion**: Twice the market level
- **Interpretation**: Conservative institutional investor (pension funds, endowments)
- **Portfolio Characteristics**: 
  - Lower expected return
  - Lower volatility
  - More diversified (closer to equal-weight)
- **Rationale**: Institutional investors often have longer time horizons and fiduciary responsibilities requiring capital preservation

**3. Kelly Investor ($\lambda_{kelly} = 1.0$)**
- **Risk Aversion**: Fixed at 1.0 (aggressive)
- **Interpretation**: Growth-oriented investor maximizing wealth
- **Portfolio Characteristics**:
  - Higher expected return
  - Higher volatility
  - More concentrated positions
- **Rationale**: Kelly criterion maximizes long-term geometric mean return, appropriate for investors with high risk tolerance

**Mathematical Relationship:**
The utility function in mean-variance optimization is:
$$U(\mathbf{w}) = \mathbf{w}'\boldsymbol{\mu} - \frac{\lambda}{2}\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}$$

Higher $\lambda$ means:
- Greater penalty for variance (risk aversion)
- More conservative portfolio (lower risk, lower return)
- Closer to minimum-variance portfolio as $\lambda \to \infty$


#### Prior Equilibrium Return Calculation

##### Mathematical Foundation: Reverse Optimization

The Black-Litterman framework derives prior equilibrium returns through reverse optimization working backwards from observed market portfolio weights to infer the return expectations that would make those weights optimal.

**Starting Point: Market Portfolio Weights**
Given market capitalization weights $\mathbf{w}_{mkt}$, we ask: "What expected returns $\boldsymbol{\mu}$ would make an investor choose these weights?"

**Optimization Problem:**
Under mean-variance optimization, the optimal portfolio solves:
$$\max_{\mathbf{w}} \mathbf{w}'\boldsymbol{\mu} - \frac{\lambda}{2}\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}$$

Subject to: $\mathbf{1}'\mathbf{w} = 1$ (fully invested)

**First-Order Condition:**
Taking the derivative with respect to $\mathbf{w}$ and setting to zero:
$$\boldsymbol{\mu} - \lambda \boldsymbol{\Sigma}\mathbf{w}_{mkt} = 0$$

**Solving for Prior Returns:**
$$\boldsymbol{\mu}_{prior} = \lambda \boldsymbol{\Sigma}\mathbf{w}_{mkt}$$

This is the **reverse optimization formula**: equilibrium returns that justify the market portfolio weights.

#### Stock Prior Returns: Reverse Optimization

For stocks, prior equilibrium returns are calculated as:

$$\boldsymbol{\mu}_{prior}^{stocks} = \lambda_{13 stock market} \times \boldsymbol{\Sigma}_{stocks} \times \mathbf{w}_{mkt}^{stocks}$$

Where:
- $\lambda_{market}$: Market risk aversion parameter
- $\boldsymbol{\Sigma}_{stocks}$: Stock covariance matrix (either conventional or Ledoit-Wolf)
- $\mathbf{w}_{mkt}^{stocks}$: Market cap weights for stocks

**Intuition:**
- Stocks with higher covariance with the market portfolio get higher equilibrium returns (risk premium)
- Stocks with larger market cap weights contribute more to market risk, requiring higher returns to justify their weight
- The formula ensures that if an investor had these return expectations, they would choose the market portfolio

##### Factor Prior Returns: Historical Mean (Alternative Approach)

**Why Factors Are Treated Differently:**

Factors have zero market cap weights ($\mathbf{w}_{mkt}^{factors} = \mathbf{0}$)

When we partition the reverse optimization formula for the combined stock-factor system:
$$\begin{bmatrix} \boldsymbol{\mu}{prior}^{stocks} \\ \boldsymbol{\mu}{prior}^{factors} \end{bmatrix} = \lambda \begin{bmatrix} \boldsymbol{\Sigma}{stocks,stocks} & \boldsymbol{\Sigma}{stocks,factors} \\ \boldsymbol{\Sigma}{factors,stocks} & \boldsymbol{\Sigma}{factors,factors} \end{bmatrix} \begin{bmatrix} \mathbf{w}{mkt}^{stocks} \\ \mathbf{0} \end{bmatrix}$$
Expanding the matrix multiplication:
$$\boldsymbol{\mu}{prior}^{factors} = \lambda \left[ \boldsymbol{\Sigma}{factors,stocks} \quad \boldsymbol{\Sigma}{factors,factors} \right] \begin{bmatrix} \mathbf{w}{mkt}^{stocks} \\ \mathbf{0} \end{bmatrix}$$
$$= \lambda \left( \boldsymbol{\Sigma}{factors,stocks} \mathbf{w}{mkt}^{stocks} + \boldsymbol{\Sigma}{factors,factors} \mathbf{0} \right)$$
$$= \lambda \boldsymbol{\Sigma}{factors,stocks} \mathbf{w}{mkt}^{stocks}$$

Interpretation:
Factor prior returns depend on the cross-covariance between factors and stocks ($\boldsymbol{\Sigma}{factors,stocks}$), not just factor-factor covariance.
The contribution is weighted by stock market cap weights ($\mathbf{w}{mkt}^{stocks}$).
Even with zero factor weights, factors still receive prior returns through their correlation with the stock portfolio.

Using historical means is preferred because:

a) Covariance-Dependent vs. Premium-Based:
Reverse optimization: Returns depend on the 13-stock portfolio's covariance structure
Historical means: Direct measure of factor premia over time
Impact: Reverse-optimized returns may not reflect true structural factor premia

b) Portfolio-Specific Bias:
Reverse optimization: Returns are specific to the 13-stock portfolio
Historical means: Reflect broader market factor premia
Impact: If the 13-stock portfolio has unusual factor exposures, reverse-optimized returns could be biased


Quantitative Comparison:
If we compare the two approaches:
Reverse Optimization: $\boldsymbol{\mu}{prior}^{factors} = \lambda \boldsymbol{\Sigma}{factors,stocks} \mathbf{w}{mkt}^{stocks}$
Could range wildly depending on portfolio composition. Highly sensitive to the 13-stock portfolio's factor exposures
Historical Means: $\boldsymbol{\mu}{prior}^{factors} = \bar{\mathbf{r}}{factors} \times 12$ (monthly, over the same calibration window the views are estimated on)
Single-digit annualized premia — stable, and consistent with the empirical factor literature


The formula shows that factors would have non-zero prior returns from reverse optimization, but these returns are:
Covariance-dependent rather than premium-based
Portfolio-specific rather than general
Indirect rather than direct observations

Therefore, using historical mean returns remains the preferred approach for factor prior returns, as it provides more stable, economically meaningful, and empirically grounded estimates of factor premia.

##### Combined Prior Return Vector

The complete prior return vector combines both approaches:

$$\boldsymbol{\mu}_{prior} = \begin{bmatrix} \boldsymbol{\mu}_{prior}^{stocks} \\ \boldsymbol{\mu}_{prior}^{factors} \end{bmatrix} = \begin{bmatrix} \lambda \boldsymbol{\Sigma}_{stocks} \mathbf{w}_{mkt}^{stocks} \\ \bar{\mathbf{r}}_{factors} \times 12 \end{bmatrix}$$


- Stocks: Returns derived from market equilibrium (reverse optimization)
- Factors: Returns derived from historical averages (empirical estimation), measured over the same window as the views so that $\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior}$ isolates the regime effect
- Consistency: Both represent "unconditional" expected returns before incorporating tactical views

##### Why This Hybrid Approach is Preferred

1. Stocks: Market cap weights provide observable equilibrium information → reverse optimization is appropriate
2. Factors: No observable market weights → historical mean is the best available estimator
3. Framework Flexibility: Black-Litterman allows different methods for different asset classes as long as they represent prior beliefs
4. Posterior Integration: Both prior estimates are updated by tactical views through the Black-Litterman formula, ensuring consistency

Connection to Black-Litterman:
The prior return vector $\boldsymbol{\mu}_{prior}$ becomes the input to the Black-Litterman formula:
$$\boldsymbol{\mu}_{BL} = \boldsymbol{\mu}_{prior} + \tau \boldsymbol{\Sigma} \mathbf{P}'(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}' + \boldsymbol{\Omega})^{-1}(\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior})$$

Where tactical views $\mathbf{Q}$ adjust the prior returns based on current macro conditions, weighted by confidence levels $\boldsymbol{\Omega}$.

### Posterior Return Calculation

#### Q Vector: Expected Returns from Tactical Views

##### Purpose

The Q vector holds one expected return per view. With one view per factor, Q has three elements — SMB, HML and MOM — each an annualized expected return.

##### Construction

Each element of Q is the **conditional historical mean** of that factor within the regime cell occupied at the view-formation date:

$$Q_i = \bar{r}_{i \mid s^*} \times 12$$

where $s^*$ is the regime state prevailing at the view date on the grid assigned to factor $i$, and $\bar{r}_{i \mid s^*}$ is the mean monthly return of factor $i$ across all calibration months classified into that state.

This is a backward-looking average applied forward, not a forecast. The claim it makes is conditional and deliberately modest: in past periods resembling the present one along the chosen macroeconomic axes, this factor earned this average premium.

##### Reading Q against the unconditional mean

Q on its own is not interpretable. A large $Q_i$ may simply reflect the factor's unconditional premium rather than anything the regime classification contributed. The informative quantity is the **regime tilt**:

$$\text{tilt}_i = Q_i - \pi_i$$

where $\pi_i$ is the factor's unconditional mean over the same calibration window. A tilt near zero means the conditioning has added nothing, and the view should be recognised as empty however large $Q_i$ appears in isolation.

##### Properties of this construction

| Property | Consequence |
|---|---|
| Q is an estimated sample mean | Its standard error is directly measurable, and becomes $\Omega$ |
| Estimated on regime subsets | Sparse regimes yield wide standard errors and therefore weak views, automatically |
| No functional form imposed | No linearity or constant-slope assumption linking macro state to factor return |
| Factor-to-grid assignment fixed in advance | The number is not selected from among many candidates after inspection |

##### Superseded construction

An earlier specification regressed factor returns on normalized macroeconomic Z-scores and formed Q as $\alpha + \sum_k \beta_k z_k$, annualized. It is retained in the code notebook, marked superseded, together with its diagnosis: persistent regressors collapse the effective sample size, published macro *levels* stand in for the surprises that would actually move returns, and a constant slope contradicts the regime-dependence premise the project is built on.

---

#### P Matrix: Factor Exposure Mapping

##### Purpose

The P (pick) matrix states which assets each view refers to. It carries one row per view and one column per asset.

##### Structure

- **Rows**: one per view — SMB, HML, MOM
- **Columns**: one per asset — 13 stocks + 3 factors = 16
- **Values**: zero on every stock, one in the column of the factor the view targets

| View | AAPL | AVGO | … | XOM | TJX | SMB | HML | MOM |
|---|---|---|---|---|---|---|---|---|
| SMB | 0 | 0 | … | 0 | 0 | **1** | 0 | 0 |
| HML | 0 | 0 | … | 0 | 0 | 0 | **1** | 0 |
| MOM | 0 | 0 | … | 0 | 0 | 0 | 0 | **1** |

##### Why every stock entry is zero

A view is a statement about a factor, so P names the factor. Stocks are not excluded from the update — they are reached through $\boldsymbol{\Sigma}$. The posterior adjustment for every asset is proportional to $\boldsymbol{\Sigma}\mathbf{P}'$, whose stock rows are precisely the **stock-factor covariance blocks**. A stock covarying strongly with HML moves substantially on an HML view; one that does not, barely moves.

This separates **view specification** (P and Q) from **risk transmission** ($\boldsymbol{\Sigma}$).

##### A consequence worth stating explicitly

Because P selects the factor directly, each stock's tilt is proportional to $\text{Cov}(r_i, f)$ — equivalently to its **univariate** beta on that factor, since the factor's variance is common across stocks. This is not the multivariate loading reported in the asset regression, which holds the other factors constant. The two differ by leakage terms scaled by the correlations among the factors themselves.

Any claim about which names a factor view goes long or short therefore has to be checked against univariate betas, which the covariance matrix already contains:

$$\hat{\beta}^{\,univ}_{i,f} = \frac{\text{Cov}(r_i, f)}{\text{Var}(f)}$$

##### Role in the posterior

$$\boldsymbol{\mu}_{BL} = \boldsymbol{\mu}_{prior} + \tau\boldsymbol{\Sigma}\mathbf{P}'\big(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}' + \boldsymbol{\Omega}\big)^{-1}\big(\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior}\big)$$

- $\mathbf{P}\boldsymbol{\mu}_{prior}$ — the prior's own expectation for each viewed factor
- $\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior}$ — the surprise the view carries relative to equilibrium
- $\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}'$ — prior uncertainty projected into view space
- $\boldsymbol{\Sigma}\mathbf{P}'$ — propagates the adjustment from the factors out to every stock

---

#### Omega Matrix: View Uncertainty and Confidence

##### Purpose

$\boldsymbol{\Omega}$ states how uncertain each view is. It is the counterweight to the prior in the posterior blend: a larger $\Omega$ means less confidence in the view and a posterior that stays closer to equilibrium.

##### Specification: the standard error of the view itself

Each view is an estimated conditional mean, so its uncertainty is the estimation error of that mean. Views are treated as independently estimated, so $\boldsymbol{\Omega}$ is diagonal:

$$\boldsymbol{\Omega} = \text{diag}\big(SE_{cluster,1}^{2}, \; \dots, \; SE_{cluster,k}^{2}\big)$$

where $SE_{cluster,i}$ is the episode-clustered standard error of view $i$, computed across contiguous-episode means rather than across months, as developed in *Conditional Means, Episode Clustering, and View Uncertainty*:

$$SE_{cluster} = \frac{s(\bar{r}_1, \dots, \bar{r}_J)}{\sqrt{J}} \times 12$$

$\Omega_{ii}$ is then the squared standard error of a quantity that was actually estimated — an interpretable number carrying the same units as the view.

##### Why not calibrate Ω from the covariance matrix

The common alternative sets $\boldsymbol{\Omega} = \text{diag}(\mathbf{P}\,\tau\,\boldsymbol{\Sigma}\,\mathbf{P}')$. It is convenient and requires no separate uncertainty estimate, but it has three drawbacks in this setting.

**It measures the wrong thing.** $\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}'$ is asset return covariance scaled by $\tau$. It assigns an uncertain view to a volatile factor and a confident view to a stable one, irrespective of how well or badly the view was actually estimated. A view resting on three episodes and a view resting on thirty receive identical treatment if their factors happen to be equally volatile.

**It introduces a $\tau$ dependence that should not exist.** Building $\Omega$ out of $\tau\boldsymbol{\Sigma}$ makes view uncertainty a function of $\tau$. If the $\tau$ used to construct $\Omega$ differs from the $\tau$ used in the posterior, the algebraic cancellation that motivates the calibration no longer holds and the prior/view balance is altered silently. A measured standard error contains no $\tau$, so the question cannot arise.

**It cannot express weak evidence.** Sparse regimes ought to produce weak views. Covariance-derived $\Omega$ offers no channel through which sample size can enter.

##### What follows from the specification

| | Covariance-derived $\Omega$ | Clustered-SE $\Omega$ |
|---|---|---|
| Depends on $\tau$ | Yes | No |
| Depends on the $\Sigma$ estimator | Yes | No |
| Responds to number of episodes | No | Yes |
| Units | Scaled asset variance | Squared annualized return |

Two consequences are worth anticipating.

First, $\Omega$ **widens substantially**. Posteriors sit closer to the prior and the resulting tilts shrink. This is the specification behaving correctly, not a loss of quality — the earlier calibration was reporting confidence the data does not contain.

Second, because $\Omega$ no longer contains $\boldsymbol{\Sigma}$, it becomes **identical across the conventional and Ledoit-Wolf tracks**. The covariance choice still affects the prior, the term $\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}'$ and the optimizers, but no longer the view uncertainty itself.

##### The diagonal assumption

Off-diagonal entries of $\Omega$ would encode correlation between the *errors* of the views. Treating $\Omega$ as diagonal asserts that the views are independently estimated. That is not automatic here: the grids share the growth axis, and the policy axis is constructed from inflation. The assumption is therefore checked by cross-tabulating the grids and reporting the association, rather than assumed.

##### Reading the result

The single most informative summary is the fraction of each posterior expectation that comes from the view rather than the prior:

$$w_{view,i} = \frac{\big(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}'\big)_{ii}}{\big(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}'\big)_{ii} + \Omega_{ii}}$$

A low view weight is not a failure of the framework. It is the framework correctly reporting that the evidence does not justify a large departure from equilibrium weights.



### Posterior Returns Calculation: Meucci's Black-Litterman Formula

#### The τ Parameter

A single $\tau$ appears in this formulation, and it has one job: to express how uncertain the
**prior** is.

The often-quoted calibration $\tau = 1/T$ is a special case, not a definition. It follows from
treating the prior as a **historical sample mean**: the sampling covariance of a mean estimated
from $T$ observations is $\boldsymbol{\Sigma}/T$ — provided $T$ is counted in the same time
unit as $\boldsymbol{\Sigma}$. Neither condition holds in this project:

- the prior is **reverse-optimized** from market-cap weights, not estimated as a sample mean, so
  there is no $T$ whose sampling error it carries;
- $\boldsymbol{\Sigma}$ is annualized, so even where a sample-mean argument applies, dividing
  by a count of *days* mixes time units and understates $\tau$ by a factor of roughly 252.

With an equilibrium prior, $\tau$ therefore has no derived value. It is set as a **stated
confidence parameter** — $\tau = 0.025$, inside the 0.01–0.05 range conventional in the
Black-Litterman literature — and disclosed as a choice rather than presented as a measurement.
Its effect is transparent: $\tau$ scales the prior-uncertainty term
$\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}'$ against a fixed, measured
$\boldsymbol{\Omega}$, so a larger $\tau$ moves the posterior toward the views and a smaller
one toward the prior. The view-weight diagnostic defined in the $\Omega$ section states the
resulting prior/view split directly, which is what makes a conventional $\tau$ acceptable: its
consequence is reported, not hidden.

Earlier drafts of this project carried **two** scaling parameters — a `tau_base` of 0.05 used to
build $\boldsymbol{\Omega}$, and $\tau = 1/T$ used in the posterior. That split is a known
inconsistency. The standard calibration $\boldsymbol{\Omega} = \text{diag}(\mathbf{P}\tau
\boldsymbol{\Sigma}\mathbf{P}')$ is motivated by an algebraic cancellation that only holds when
both $\tau$ values are the same, so using different ones shifts the prior/view balance without
declaring it. Deriving $\boldsymbol{\Omega}$ from the views' own standard errors removed the
second parameter entirely; replacing $1/T$ with the stated constant then fixed the remaining
$\tau$'s meaning, for the estimator and unit reasons above. One $\tau$, one meaning, one place.

#### Meucci's Black-Litterman Posterior Return Formula

##### The Complete Formula

$$\boldsymbol{\mu}_{BL} = \boldsymbol{\mu}_{prior} + (\tau \boldsymbol{\Sigma}) \mathbf{P}'(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}' + \boldsymbol{\Omega})^{-1}(\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior})$$

##### Formula Breakdown: Component-by-Component

The formula can be understood as:

**Posterior Mean = Prior Mean + Prior Uncertainty × Asset-View Link × Confidence × Surprise**



1. Prior Mean (μ_prior)

Component: $\boldsymbol{\mu}_{prior}$

Interpretation:
- Starting point: equilibrium returns from reverse optimization (stocks) or historical means (factors)
- Represents expected returns before incorporating tactical views
- The "base case" return expectations

In This Implementation:
- Stocks: Derived from reverse optimization ($\lambda \boldsymbol{\Sigma}_{stocks} \mathbf{w}_{mkt}^{stocks}$)
- Factors: Unconditional mean over the **same calibration window the views are estimated on** ($\bar{\mathbf{r}}_{factors} \times 12$, monthly data) — so the surprise term $\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior}$ is exactly the regime tilt, not a comparison across two different sample windows

2. Prior Uncertainty (τΣ)

Component: $\tau \boldsymbol{\Sigma}$

Interpretation:
- Scaled prior covariance matrix
- Represents uncertainty in the prior returns
- τ = 0.025, a stated prior-confidence constant (see *The τ Parameter* above)

Role:
- τΣ sets how far the posterior is allowed to move from the prior in response to the views

Mathematical Meaning:
- Covariance scaled by the uncertainty parameter
- Reflects how much we trust (or don't trust) the prior estimates

3. Asset-View Link (P')

Component: $\mathbf{P}'$ (transpose of P matrix)

Interpretation:
- Maps view impacts back to asset space
- Connects factor views to asset returns
- Transposes the P matrix to propagate view effects

Role:
- If P maps assets to views, then P' maps views back to assets
- Links factor-level views to individual asset returns
- Essential for propagating factor views to stocks

In This Implementation:
- P has zeros for stocks, ones for factors
- P' allows factor views to influence stock returns through the covariance structure
- Creates the connection between view space and asset space

4. Confidence (Middle Term Inverse)

Component: $(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}' + \boldsymbol{\Omega})^{-1}$

Interpretation:
- Inverse of the combined uncertainty (prior + views)
- Acts as a "precision" or "confidence" matrix
- Higher values → more confidence → more weight to views
- Lower values → less confidence → more weight to prior

Breaking Down the Components:

a) PτΣP':
- Prior uncertainty in the view space
- Covariance of views based on prior
- Measures how views covary based on asset structure

b) Ω:
- Uncertainty in the views themselves — the squared standard error of each estimated conditional mean
- Higher Ω → less confidence in views → posterior stays near the prior
- Lower Ω → more confidence in views → posterior moves toward Q

c) Sum and Inverse:
- Combined uncertainty = Prior uncertainty + View uncertainty
- Inverse = Precision = Confidence
- $(Uncertainty)^{-1} = Confidence$

Role:
- Weights the surprise term based on confidence
- If views are confident (low uncertainty) → large inverse → views weighted heavily
- If views are uncertain (high uncertainty) → small inverse → prior weighted heavily

5. Surprise (Q - Pμ_prior)

Component: $(\mathbf{Q} - \mathbf{P}\boldsymbol{\mu}_{prior})$

Interpretation:
- Difference between view expectations and prior expectations
- "Surprise" or "innovation" from tactical views
- Measures how views differ from prior equilibrium

Breaking Down:

a) Q:
- Tactical view expected returns
- Forward-looking expectations from macro analysis
- What we "think" returns should be

b) Pμ_prior:
- Prior expected returns in view space
- What the prior "thinks" factor returns should be
- Prior expectations projected onto view space

c) Difference (Q - Pμ_prior):
- The "surprise" or deviation
- Positive → views are more optimistic than prior
- Negative → views are more pessimistic than prior
- Zero → views agree with prior (no adjustment needed)

Role:
- This is the "signal" that drives adjustments
- Larger surprise → larger adjustments
- Smaller surprise → smaller adjustments

##### How the Components Work Together

Step-by-Step Process:

1. Calculate Surprise: $(Q - P\mu_{prior})$ tells us how views differ from prior

2. Weight by Confidence: $(\mathbf{P}\tau\boldsymbol{\Sigma}\mathbf{P}' + \boldsymbol{\Omega})^{-1}$ determines how much to trust this surprise

3. Map to Asset Space: $\mathbf{P}'$ propagates the weighted surprise to assets

4. Scale by Uncertainty: $(\tau \boldsymbol{\Sigma})$ determines the magnitude of impact

5. Add to Prior: $\boldsymbol{\mu}_{prior} + ...$ combines prior with adjustments

Mathematical Flow:
```
Prior → Surprise → Confidence Weighting → Asset Mapping → Uncertainty Scaling → Posterior
```

Intuitive Understanding:
- Start with prior (equilibrium) returns
- Calculate how much views differ (surprise)
- Weight by confidence (inverse uncertainty)
- Map to assets (P')
- Scale by uncertainty (τΣ)
- Add adjustment to prior → Posterior returns

##### Example Interpretation

If Surprise is Large (e.g., Q = 20%, Pμ_prior = 5%):
- Surprise = 15% (very optimistic view)
- If confidence is high (low Ω) → Large inverse → Large weight
- Adjustment = τΣ × P' × Large weight × 15%
- Result: Significant upward adjustment to returns

If Surprise is Small (e.g., Q = 6%, Pμ_prior = 5%):
- Surprise = 1% (view close to prior)
- Even with high confidence → Small surprise → Small adjustment
- Result: Minimal change to returns

If Confidence is Low (High Ω):
- Even with large surprise → Small inverse → Small weight
- Adjustment is dampened
- Result: Prior dominates, views have minimal impact

---

#### Posterior Covariance Formula: Overview

##### The Formula

$$\boldsymbol{\Sigma}_{BL} = (1 + \tau) \boldsymbol{\Sigma}_{prior} - (\tau^2 \boldsymbol{\Sigma}_{prior}) \mathbf{P}'(\mathbf{P}\tau\boldsymbol{\Sigma}_{prior}\mathbf{P}' + \boldsymbol{\Omega})^{-1} \mathbf{P} \boldsymbol{\Sigma}_{prior}$$

##### Intuitive Understanding

Structure:
- First Term: $(1 + \tau) \boldsymbol{\Sigma}_{prior}$ - Scaled prior covariance (increased uncertainty)
- Second Term: $(\tau^2 \boldsymbol{\Sigma}_{prior}) \mathbf{P}'(\mathbf{P}\tau\boldsymbol{\Sigma}_{prior}\mathbf{P}' + \boldsymbol{\Omega})^{-1} \mathbf{P} \boldsymbol{\Sigma}_{prior}$ - Uncertainty reduction from views

Key Insight:
- Posterior covariance = Prior covariance + Uncertainty increase - Uncertainty reduction
- Views reduce uncertainty (we learn from them)
- But τ scaling increases uncertainty (we're less certain about priors)

Why (1 + τ) Scaling?
- Represents increased uncertainty due to uncertainty in prior
- Standard approach in Black-Litterman to account for prior uncertainty
- Ensures posterior covariance reflects both prior and view uncertainties

Why Subtract the Second Term?
- Views provide information → reduce uncertainty
- The term $(\mathbf{P}\tau\boldsymbol{\Sigma}_{prior}\mathbf{P}' + \boldsymbol{\Omega})^{-1}$ represents information gain
- Subtracting it reflects that we're more certain after incorporating views
- Larger information gain → larger reduction in uncertainty

##### Relationship to Posterior Returns

Consistency:
- Both formulas use the same middle term: $(\mathbf{P}\tau\boldsymbol{\Sigma}_{prior}\mathbf{P}' + \boldsymbol{\Omega})^{-1}$
- Both use τ to scale uncertainty
- Both use P to map between view space and asset space

Key Difference:
- Posterior returns: **Add** adjustments (views change expected returns)
- Posterior covariance: **Subtract** uncertainty (views reduce uncertainty)

Interpretation:
- Views change what we expect (returns)
- Views also change how certain we are (covariance)
- More confident views → larger return adjustments + larger uncertainty reduction

##### Impact on Portfolio Optimization

Posterior Returns:
- Determine expected returns for optimization
- Drive portfolio tilts and allocations
- Represent the "mean" in mean-variance optimization

Posterior Covariance:
- Determine risk estimates for optimization
- Drive diversification and risk management
- Represent the "variance" in mean-variance optimization

Together:
- Provide the inputs for portfolio optimization
- Balance return expectations with risk estimates
- Create the efficient frontier for portfolio construction

---


# 4. Portfolio Optimizations

### Mean Variance


#### Basic Concept

Mean-variance optimization (MVO) is a portfolio construction framework that seeks to maximize expected return for a given level of risk (Institutional setting), or minimize risk for a given level of expected return (Retail Investors). It's based on Markowitz's Modern Portfolio Theory.

#### Core Principle

**The Efficient Frontier:**
- Investors prefer portfolios with higher expected returns and lower risk
- The efficient frontier represents the set of portfolios that offer the highest expected return for each level of risk
- Any portfolio below the efficient frontier is suboptimal (can get higher return for same risk, or lower risk for same return)

#### The Optimization Problem

**Objective Function:**
Maximize utility, where utility is a trade-off between expected return and risk:

$$\max_{\mathbf{w}} \quad U(\mathbf{w}) = \mathbf{w}'\boldsymbol{\mu} - \frac{\lambda}{2}\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}$$


#### Constraints

**Constraints:**
1. **Fully Invested:** $\sum_{i=1}^{n} w_i = 1$ (Stock weights sum to 1)
2. **No Short Selling:** $w_i \geq 0$ for all $i$ (long-only constraints for Stocks)
3. **Factor Constraints:** $\sum_{j \in factors} w_j = 0$ (factor weights sum to zero)

#### First-Order Conditions

Taking the derivative with respect to **w** and setting to zero:

$$\frac{\partial U}{\partial \mathbf{w}} = \boldsymbol{\mu} - \lambda \boldsymbol{\Sigma}\mathbf{w} = 0$$

Solving for optimal weights:

$$\boldsymbol{\mu} = \lambda \boldsymbol{\Sigma}\mathbf{w}^*$$

$$\mathbf{w}^* = \frac{1}{\lambda}\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$$

**Interpretation:**
- Optimal weights are proportional to $\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$
- Inverse covariance matrix "decorrelates" returns
- Higher expected returns → Higher weights
- Higher covariance (risk) → Lower weights



#### Why we get similar weights for Ledoit-Wolf and BL Posterior Covariance?

Mean-variance optimization is primarily driven by expected returns (μ), not covariance structure (Σ)

##### Mathematical Explanation

**The Optimal Weight Formula:**

$$\mathbf{w}^* = \frac{1}{\lambda}\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$$

**Key Observations:**

1. **Covariance appears only in the inverse: $\boldsymbol{\Sigma}^{-1}$**
   - The inverse covariance matrix acts as a "decorrelation" or "risk adjustment" factor
   - It scales returns by their risk-adjusted attractiveness
   - But the **direction** of weights is primarily determined by **μ**

2. **Expected returns (μ) are the same:**
   - Both optimizations use the **same posterior returns** ($\boldsymbol{\mu}_{BL}$)
   - The covariance structure doesn't change expected returns
   - Therefore, the "signal" (expected returns) is identical

3. **Covariance differences are "normalized" by the inverse:**
   - Ledoit-Wolf: Shrinks covariance toward structured target
   - BL Posterior: Adjusts covariance based on views
   - But when inverted, both provide similar risk adjustments
   - The inverse operation tends to "smooth out" differences

##### Why the Inverse Smooths Differences

**Covariance Matrix Properties:**

1. **Inverse Operation:**
   - $\boldsymbol{\Sigma}^{-1}$ is the precision matrix
   - Inverting tends to reduce the impact of small differences
   - Both matrices are positive definite and well-conditioned

2. **Similar Structure:**
   - Both Ledoit-Wolf and BL posterior covariance maintain similar correlation structures
   - The differences are primarily in the magnitude of variances, not correlations
   - When inverted, these magnitude differences are normalized

3. **Risk Adjustment Effect:**
   - The inverse covariance essentially "risk-adjusts" the returns
   - $\boldsymbol{\Sigma}^{-1}\boldsymbol{\mu}$ gives risk-adjusted expected returns
   - Both covariance structures provide similar risk adjustments

##### Numerical Example

**Suppose we have two covariance matrices:**

**Ledoit-Wolf:**
$$\boldsymbol{\Sigma}_{LW} = \begin{bmatrix} 0.04 & 0.02 \\ 0.02 & 0.05 \end{bmatrix}$$

**BL Posterior:**
$$\boldsymbol{\Sigma}_{BL} = \begin{bmatrix} 0.042 & 0.021 \\ 0.021 & 0.052 \end{bmatrix}$$

**Expected Returns:**
$$\boldsymbol{\mu} = \begin{bmatrix} 0.10 \\ 0.12 \end{bmatrix}$$

**Inverse Matrices:**

$$\boldsymbol{\Sigma}_{LW}^{-1} = \begin{bmatrix} 31.25 & -12.5 \\ -12.5 & 25.0 \end{bmatrix}$$

$$\boldsymbol{\Sigma}_{BL}^{-1} = \begin{bmatrix} 30.30 & -12.12 \\ -12.12 & 24.24 \end{bmatrix}$$

**Risk-Adjusted Returns:**

$$\boldsymbol{\Sigma}_{LW}^{-1}\boldsymbol{\mu} = \begin{bmatrix} 1.25 \\ 1.5 \end{bmatrix}$$

$$\boldsymbol{\Sigma}_{BL}^{-1}\boldsymbol{\mu} = \begin{bmatrix} 1.21 \\ 1.45 \end{bmatrix}$$

**Result:**
- The risk-adjusted returns are very similar (within 3-4%)
- After normalization by λ, the weights will be nearly identical
- Small differences in covariance lead to small differences in weights


Mean-variance optimization is primarily a return-seeking exercise, with covariance providing risk adjustment. When expected returns are identical, different but reasonable covariance estimates will produce similar optimal weights


### Sharpe Ratio


#### Basic Concept

Sharpe Ratio optimization seeks to maximize the **risk-adjusted return** of a portfolio. Unlike mean-variance optimization which trades off return and risk linearly, Sharpe ratio optimization maximizes the **ratio** of excess return to volatility.

#### Core Principle

**The Sharpe Ratio:**
- Measures risk-adjusted performance: $SR = \frac{\mu_p - r_f}{\sigma_p}$
- Higher Sharpe ratio → Better risk-adjusted returns
- Maximizing Sharpe ratio finds the portfolio on the efficient frontier with the steepest risk-return trade-off
- This is the **tangency portfolio** (portfolio with maximum Sharpe ratio)

#### The Optimization Problem

**Objective Function:**
Maximize the Sharpe Ratio:

$$\max_{\mathbf{w}} \quad SR(\mathbf{w}) = \frac{\mathbf{w}'\boldsymbol{\mu} - r_f}{\sqrt{\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}}}$$


#### Constraints

**Typical Constraints:**
1. **Fully Invested:** $\sum_{i=1}^{n} w_i = 1$ (Stock weights sum to 1)
2. **No Short Selling:** $w_i \geq 0$ for all $i$ (long-only constraint for Stocks)
3. **Factor Constraints:** $\sum_{j \in factors} w_j = 0$ (Factor weights sum to zero)

#### Optimization Approach

**Numerical Optimization:**
- No closed-form solution (unlike mean-variance)
- Minimize negative Sharpe ratio: $\min_{\mathbf{w}} \quad -\frac{\mathbf{w}'\boldsymbol{\mu}}{\sqrt{\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}}}$



#### Why Weights Differ: Conventional vs. Ledoit-Wolf Covariance

Sharpe Ratio optimization is highly sensitive to covariance structure because covariance appears in the denominator (volatility), creating a nonlinear relationship.

##### Mathematical Explanation

**The Sharpe Ratio Formula:**

$$SR = \frac{\mathbf{w}'\boldsymbol{\mu}}{\sqrt{\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}}}$$

**Why Covariance Matters More:**

1. **Covariance in Denominator:**
   - Volatility = $\sqrt{\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}}$ appears in the denominator
   - Changes in Σ directly affect the Sharpe ratio through the denominator
   - **Nonlinear impact**: Small changes in Σ can have large impacts on the ratio

2. **Nonlinear Relationship:**
   - Unlike mean-variance (linear in Σ), Sharpe ratio is **nonlinear**
   - The ratio form amplifies the impact of covariance differences
   - Optimization is more sensitive to covariance estimation errors

3. **Volatility Optimization:**
   - Maximizing Sharpe ratio is equivalent to minimizing volatility for a given return
   - Or maximizing return for a given volatility
   - Covariance directly determines volatility, so it's crucial

##### Why Conventional vs. Ledoit-Wolf Differ

**1. Estimation Error in Conventional Covariance:**

**Conventional Sample Covariance:**
- Uses standard sample estimator: $\hat{\Sigma} = \frac{1}{T-1}\sum_{t=1}^{T}(\mathbf{r}_t - \bar{\mathbf{r}})(\mathbf{r}_t - \bar{\mathbf{r}})'$
- **Problems:**
  - High estimation error when $n$ (assets) is large relative to $T$ (observations)
  - Extreme eigenvalues (very large and very small)
  - Can be ill-conditioned or near-singular
  - Sensitive to outliers

**Impact on Sharpe Ratio:**
- Estimation errors in Σ lead to **incorrect volatility estimates**
- Incorrect volatility → Incorrect Sharpe ratio calculation
- Optimization may find suboptimal weights based on wrong risk estimates
- Portfolio may appear to have better Sharpe ratio than it actually does

**2. Ledoit-Wolf Shrinkage Benefits:**

**Ledoit-Wolf Covariance:**
- Shrinks sample covariance toward structured target: $\Sigma_{LW} = \alpha \Sigma_{target} + (1-\alpha)\Sigma_{sample}$
- **Benefits:**
  - Reduces estimation error
  - Improves conditioning (more stable eigenvalues)
  - More robust to outliers
  - Better out-of-sample performance

**Impact on Sharpe Ratio:**
- More accurate volatility estimates
- More stable optimization
- Better risk assessment
- Portfolio weights reflect true risk structure

##### Numerical Example: Why Differences Matter

**Suppose we have two covariance matrices:**

**Conventional (with estimation error):**
$$\boldsymbol{\Sigma}_{conv} = \begin{bmatrix} 0.04 & 0.025 \\ 0.025 & 0.06 \end{bmatrix}$$

**Ledoit-Wolf (shrunk, more stable):**
$$\boldsymbol{\Sigma}_{LW} = \begin{bmatrix} 0.038 & 0.022 \\ 0.022 & 0.055 \end{bmatrix}$$

**Expected Returns:**
$$\boldsymbol{\mu} = \begin{bmatrix} 0.10 \\ 0.12 \end{bmatrix}$$

**Portfolio Weights (example):**
$$\mathbf{w} = \begin{bmatrix} 0.6 \\ 0.4 \end{bmatrix}$$

**Portfolio Volatilities:**

**Conventional:**
$$\sigma_{conv} = \sqrt{\mathbf{w}'\boldsymbol{\Sigma}_{conv}\mathbf{w}} = \sqrt{0.6^2 \times 0.04 + 2 \times 0.6 \times 0.4 \times 0.025 + 0.4^2 \times 0.06} = 0.197$$

**Ledoit-Wolf:**
$$\sigma_{LW} = \sqrt{\mathbf{w}'\boldsymbol{\Sigma}_{LW}\mathbf{w}} = \sqrt{0.6^2 \times 0.038 + 2 \times 0.6 \times 0.4 \times 0.022 + 0.4^2 \times 0.055} = 0.186$$

**Sharpe Ratios:**

**Conventional:**
$$SR_{conv} = \frac{0.6 \times 0.10 + 0.4 \times 0.12}{0.197} = \frac{0.108}{0.197} = 0.548$$

**Ledoit-Wolf:**
$$SR_{LW} = \frac{0.6 \times 0.10 + 0.4 \times 0.12}{0.186} = \frac{0.108}{0.186} = 0.581$$

**Result:**
- **6% difference in volatility** leads to **6% difference in Sharpe ratio**
- Optimization will find different weights to maximize each Sharpe ratio
- The portfolio that maximizes $SR_{conv}$ will differ from the one maximizing $SR_{LW}$



For Sharpe ratio optimization, we use Ledoit-Wolf (or BL posterior) covariance rather than conventional sample covariance.

### Equal Risk Contribution


#### Basic Concept

Equal Risk Contribution (ERC), also known as **Risk Parity**, seeks to construct a portfolio where each asset contributes equally to the portfolio's total risk. Unlike mean-variance or Sharpe ratio optimization which focus on returns, ERC focuses purely on **risk diversification** by equalizing risk contributions across assets.

#### Core Principle

**The Risk Parity Philosophy:**
- Each asset should contribute the same amount of risk to the portfolio
- Risk contributions are equalized, not returns or weights
- Achieves true diversification by balancing risk, not capital
- Higher volatility assets get lower weights, lower volatility assets get higher weights
- This is the **risk parity portfolio** (portfolio with equal risk contributions)


#### The Optimization Problem

**Objective Function:**
Minimize the variance of risk contributions across all assets:

$$\min_{\mathbf{w}} \quad f(\mathbf{w}) = \sum_{i=1}^{n} \sum_{j=1}^{n} (RC_i - RC_j)^2$$

Where:
- **w** = Vector of portfolio weights (decision variables)
- **RC_i** = Risk Contribution of asset $i$
- **n** = Number of assets

#### Risk Contribution Components

**1. Portfolio Volatility:**
$$\sigma_p = \sqrt{\mathbf{w}'\boldsymbol{\Sigma}\mathbf{w}}$$

**2. Marginal Contribution to Risk (MCR):**
$$MCR_i = \frac{(\boldsymbol{\Sigma}\mathbf{w})_i}{\sigma_p} = \frac{\sum_{j=1}^{n} \sigma_{ij} w_j}{\sigma_p}$$

**Interpretation:**
- Measures how much portfolio volatility changes when asset $i$'s weight increases by a small amount
- The $i$-th element of the vector $\boldsymbol{\Sigma}\mathbf{w}$ divided by portfolio volatility
- Higher MCR → Asset contributes more to portfolio risk per unit of weight

**3. Risk Contribution (RC):**
$$RC_i = w_i \times MCR_i = \frac{w_i (\boldsymbol{\Sigma}\mathbf{w})_i}{\sigma_p}$$

**Interpretation:**
- The actual amount of portfolio volatility attributed to asset $i$
- Weight × Marginal Contribution to Risk
- Represents the share of total risk from asset $i$

**4. Euler's Theorem:**
$$\sum_{i=1}^{n} RC_i = \sigma_p$$

The sum of all risk contributions equals the total portfolio volatility.

#### The ERC Condition

**Equal Risk Contribution:**
$$RC_i = RC_j \quad \forall i, j$$

All assets contribute the same amount of risk to the portfolio.

**Implication:**
- If all $RC_i$ are equal, then: $RC_i = \frac{\sigma_p}{n}$ for all $i$
- Each asset contributes $\frac{1}{n}$ of the total portfolio risk

#### Objective Function Details

**Pairwise Difference Matrix:**
The objective function creates a matrix of all pairwise differences:

$$f(\mathbf{w}) = \sum_{i=1}^{n} \sum_{j=1}^{n} (RC_i - RC_j)^2$$

**Why Sum All Pairs (Double Counting)?**

1. **Computational Efficiency:**
   - NumPy's vectorized operations are highly optimized
   - Processing the full $n \times n$ matrix is faster than extracting unique pairs
   - BLAS/LAPACK operations on contiguous memory blocks

2. **Gradient Smoothness:**
   - Creates a symmetric, differentiable penalty surface
   - The global minimum (zero) occurs at the same weights regardless of double counting
   - Optimization algorithm benefits from consistent dimensionality

3. **Solver Stability:**
   - SLSQP algorithm benefits from consistent $n \times n$ structure
   - Easier to calculate Jacobian for optimization path
   - More stable convergence

#### Constraints

**Typical Constraints:**
1. **Fully Invested:** $\sum_{i=1}^{n} w_i = 1$ (weights sum to 1)
2. **No Short Selling:** $w_i \geq 0$ for all $i$ (long-only constraint)
3. **Note:** ERC is typically performed on stocks only (factors excluded)

#### Optimization Approach

**Numerical Optimization:**
- No closed-form solution (nonlinear optimization problem)
- Minimize the sum of squared differences in risk contributions
- Uses iterative optimization (e.g., SLSQP)
- Requires initial guess (typically equal weights: $w_i = 1/n$)

#### Appendix: Derivation of the Equal Risk Contribution (ERC) Objective

##### 1. Derivation of Marginal Contribution to Risk (MCR)

The **Total Portfolio Volatility** ($\sigma_p$) is defined as:

$$\sigma_p = \sqrt{w^\top \Sigma w}$$

To find the **Marginal Contribution to Risk** ($\text{MCR}_i$), we need to calculate the partial derivative of $\sigma_p$ with respect to a single asset weight $w_i$. We use the **Chain Rule**:

Let $V = w^\top \Sigma w$ (Portfolio Variance), so $\sigma_p = V^{1/2}$.

Step A: Derivative of Variance

The derivative of the quadratic form $w^\top \Sigma w$ with respect to the vector $w$ is $2\Sigma w$. For a single component $w_i$:

$$\frac{\partial V}{\partial w_i} = 2(\Sigma w)_i$$

Step B: Applying the Chain Rule

$$\frac{\partial \sigma_p}{\partial w_i} = \frac{\partial V^{1/2}}{\partial w_i} = \frac{1}{2} V^{-1/2} \cdot \frac{\partial V}{\partial w_i}$$

Step C: Substitution

Substitute $V^{1/2} = \sigma_p$ and $\frac{\partial V}{\partial w_i} = 2(\Sigma w)_i$:

$$\frac{\partial \sigma_p}{\partial w_i} = \frac{1}{2\sigma_p} \cdot 2(\Sigma w)_i$$

Final MCR Formula

The constants ($\frac{1}{2}$ and $2$) cancel out, leaving:

$$\text{MCR}_i = \frac{(\Sigma w)_i}{\sigma_p}$$



##### 2. Risk Contribution ($\text{RC}_i$) and the ERC Condition

The **Risk Contribution** of asset $i$ is the share of total volatility attributed to that asset:

$$\text{RC}_i = w_i \cdot \text{MCR}_i = \frac{w_i (\Sigma w)_i}{\sigma_p}$$

The **Euler Decomposition** of risk states that the sum of these contributions equals the total portfolio volatility:

$$\sum_{i=1}^n \text{RC}_i = \sigma_p$$

The **ERC Condition** is satisfied when all risk contributions are equal:

$$\text{RC}_i = \text{RC}_j \quad \forall i, j$$



##### 3. Vectorized Objective Function

To solve this numerically, we minimize the squared differences between all pairs of risk contributions.

The Pairwise Difference Matrix

Using NumPy broadcasting, we create a matrix of all possible differences $(\text{RC}_i - \text{RC}_j)$:

1. **`rc[:, np.newaxis]`**: Reshapes the 1D array ($\text{RC}$) into a 2D column vector.
2. **`rc`**: Remains a 1D row vector.
3. **Subtraction**: NumPy "broadcasts" both arrays into an $n \times n$ matrix where the element at $(i,j)$ is $\text{RC}_i - \text{RC}_j$.




---

# 5. Performance & Risk Analytics

### Tracking Error Anlaysis


#### 1. Tracking Error

Definition
**Tracking Error** measures the volatility of portfolio returns relative to benchmark returns. It quantifies how much a portfolio deviates from its benchmark.

##### Formula

```
Tracking Error = √(Active Weights' × Covariance Matrix × Active Weights)
```

Where:
- **Active Weights** = Portfolio Weights - Benchmark Weights
- **Covariance Matrix** = Ledoit-Wolf covariance matrix (annualized)


#### 2. Information Ratio

Definition
**Information Ratio** measures the excess return per unit of tracking error. It's the risk-adjusted measure of active management skill.

##### Formula
```
Information Ratio = (Portfolio Return - Benchmark Return) / Tracking Error
```

##### Annualized Calculation
```
IR = (Annualized Portfolio Return - Annualized Benchmark Return) / Annualized Tracking Error
```

##### Interpretation
- **IR > 0**: Portfolio outperforms benchmark on risk-adjusted basis
- **IR > 1**: Strong outperformance (1% excess return per 1% tracking error)
- **IR > 2**: Excellent active management
- **Higher IR = Better**: More excess return per unit of active risk


### Risk Contribution Analysis

#### Marginal Contribution to Risk (MCR)

**MCR** measures the sensitivity of portfolio volatility to a small change in the weight of asset i. It tells you how much the portfolio's total risk changes when you increase asset i's weight by a small amount.

#### Risk Contribution (RC)

**RC** measures the actual amount of portfolio volatility attributed to each asset. It's the asset's weight multiplied by its marginal contribution to risk.

#### Relationship

- **RC** = Weight × **MCR**
- Sum of all **RC** values equals the total portfolio volatility (Euler's theorem)

#### 2. Formulas

#### Marginal Contribution to Risk (MCR)

```
MCR_i = (Σ @ w)_i / σ_p
```

Where:
- `Σ` = Covariance matrix (Ledoit-Wolf: `ledoit_wolf_cov.loc[stock_list, stock_list]`)
- `w` = Portfolio weight vector
- `(Σ @ w)` = Matrix multiplication (covariance matrix × weights)
- `σ_p` = Portfolio volatility = `√(w' @ Σ @ w)`
- `(Σ @ w)_i` = i-th element of the resulting vector

#### Risk Contribution (RC)

```
RC_i = w_i × MCR_i = (w_i × (Σ @ w)_i) / σ_p
```

#### Portfolio Volatility

```
σ_p = √(w' @ Σ @ w)
```

#### Euler's Theorem

```
σ_p = Σ RC_i  (sum of all risk contributions equals portfolio volatility)
```

---